[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RickBarretto/llm-playground/blob/main/play.ipynb)

## Setup

In order to run this notebook, you need to install our own library called `ricklm`.

In [ ]:
!rm -rf /content/llm-playground
!git clone https://github.com/RickBarretto/llm-playground
!pip install /content/llm-playground

## Set Model

|Model|Available Sizes|Default Size|Hugging Face|
|:---:|:--------------|:----------:|------------|
|`AmadeusVerbo`  |`0.5B`, `1.5B`, `3B`, `7B`, `14B`, `32B`, `72B`|`7B`|https://huggingface.co/collections/amadeusai/amadeus-verbo-qwen25-pt-br-powered-by-aws|
|`Gaia`          |`4B`|`4B`|https://huggingface.co/CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it|
|`Tucano`        |`1.1B`, `2.4B`| `2.4B`|https://huggingface.co/collections/TucanoBR/tucano|
|`TeenyTinyLlama`|`460m`| `460m`|https://huggingface.co/collections/nicholasKluge/teenytinyllama|

**Usage:**

```python
from ricklm import models

models.Model(size="7B")
```

In [ ]:
from ricklm import models

model = models.AmadeusVerbo(size="7B")

## Do Task

- `Poem`:
  - `by`: `Model` - The model that will generate the poem.
  - `style`: `str` - The style the poem should have. E.g.: "Carlos Drummond"
  - `titled`: `str` - The title/theme of the poem. (Optional)
  - `knowing`: `str` - Context about the author and his style. (Optional)
  - `examples`: `list[str]` - Examples of original Poems. (Optional)

  **Usage:**

  ```python
  from ricklm import tasks

  with tasks.Task(by=model, params) as task:
      print(task)
  ```

### Single poem (quick test)

In [ ]:
from ricklm import tasks
from ricklm.features.poems import examples_at

style="Carlos Drummond de Andrade"
examples = examples_at("evaluations/drummond")

with tasks.Poem(by=model, style=style, eg=examples) as poem:
    print(poem)


### Batch evaluation

Runs every prompt in `prompts/poems/` (drummond, euclides, gregório …) **3 times** each and saves results to:

```
evaluations/<prompt_name>/<ModelName>/1.txt
evaluations/<prompt_name>/<ModelName>/2.txt
evaluations/<prompt_name>/<ModelName>/3.txt
```

**Usage**:

```py
from pathlib import Path
from ricklm.features.poems import evaluate

# We need this because of Collab's environment
REPO_ROOT = Path("/content/llm-playground")
evaluate.evaluate_all(model, times=3, root=REPO_ROOT)
```

In [ ]:
from pathlib import Path
from ricklm.features.poems import evaluate

REPO_ROOT = Path("/content/llm-playground")
evaluate.evaluate_all(model, times=3, root=REPO_ROOT)

### Push to GitHub

Commits the `evaluations/` folder and pushes to the repo.

Set `GITHUB_TOKEN` and pass your git username and email to push into your repository.

To set secrets: (🔑 sidebar → Secrets).

**Usage**

```py
from ricklm.features import GitHub, Path
from ricklm.features import secret

GitHub(
    repo="Username/Repo",
    token=secret("GITHUB_TOKEN"),
    username=secret("GIT_NAME"),
    email=secret("GIT_EMAIL"),
    branch="your-branch",
    root = REPO_ROOT
).push("Your commit message")
```

In [ ]:
from ricklm.features import GitHub, Path
from ricklm.features import secret

GitHub(
    repo="RickBarretto/llm-playground",
    token=secret("GITHUB_TOKEN"),
    username=secret("GIT_NAME"),
    email=secret("GIT_EMAIL"),
    branch="main",
    root = REPO_ROOT
).push(f"[Poems] Publish generated poem at style of {style} by {model}.")

If you want to reuse this same runtime, make sure to `mode.release()` and `del model`.

This ensures the storage will be clean and also the model will be garbage collected.

In [ ]:
model.release()
del model